# bigram 中文名字生成模型

本 notebook 是 `makemore/bigram.ipynb` 的中文版本（古人名字），使用中文姓名数据训练 bigram 模型。

# 思路想法
-1.先读取名字，看一下有多少个单独出现的字，大概确定一下词表的范围，和英文不同，中文的字要多一点

-2.词表估计会超级大所以要怎么简化运算？

-3.依旧穿件stoi和itos，然后遍历词表搞好输入输出的xs和ys即可，

In [1]:
mingzi=open('names-CHINESE.txt','r',encoding='utf-8').read().splitlines()
mingzi[0] = mingzi[0].lstrip('\ufeff')#开头有个表明中文的我剔除掉了
print(mingzi[:5])

['阿八哈', '阿巴雅', '阿班羅桑曲殿', '阿般圖', '阿保']


In [ ]:
from collections import Counter
cnt = Counter(''.join(mingzi))
print(len(cnt))   # 看看到底多少个字
#输出5505，那我们就要自己设计一下了，确实不少，得裁剪一些用不上的
keep={ch for ch, c in cnt.items() if c>=7}

mingzi_clean = [name for name in mingzi if all(ch in keep for ch in name)]
#注意后面引用的时候记得用这个
chars=sorted(list(set(''.join(mingzi_clean))))
print(len(chars))
#ok,还剩下3377，不错，接下来就是重复英文的流程了就好了


5490
3377


In [3]:
import torch
stoi={s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}
#词表构建完成， 接下来用同一套随机数，我们就不画图了，因为这个太长了，直接上神经网络法

In [9]:
import torch.nn.functional as F
xs,ys=[],[]
for w in mingzi_clean:
    chs=['.']+list(w)+['.']
    for ch1,ch2 in zip(chs,chs[1:]):
        ix1=stoi[ch1]
        ix2=stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)
xs=torch.tensor(xs)
ys=torch.tensor(ys)
num=xs.nelement()
g=torch.Generator().manual_seed(2147483647)
W=torch.randn((3378,3378),generator=g,requires_grad=True)
print(max(xs), max(ys))        # 看看最大索引是多少
print(len(stoi), len(itos))    # 看看词表实际多大
print(W.shape)                 # 看看 W 的形状

tensor(3377) tensor(3377)
3378 3378
torch.Size([3378, 3378])


In [20]:
#前向传播，但是我们这里要分批次训练，因为这个中文输入量太大了，如果全部输入就是九十四万乘以3377，我电脑会炸的
for i in range(500):
    ix=torch.randint(0,num,(30000,),generator=g)
    #不用onehot，用了会炸，我问aiai告诉我的，确实直接onehot3377分之1，就很没必要了，直接查表！
    logits=W[xs[ix]]
    #剩下就差不多了
    counts=logits.exp()
    probs=counts/counts.sum(1,keepdim=True)
    loss=-probs[torch.arange(30000),ys[ix]].log().mean()+0.1*(W**2).mean()#很关键的正则化，因为我们需要一些圆滑
    if i%50==0:
        print("目前的loss=",loss.item())
    W.grad=None
    loss.backward()
    W.data+=-100*W.grad

目前的loss= 5.991162300109863
目前的loss= 5.930225849151611
目前的loss= 5.894503116607666
目前的loss= 5.874702453613281
目前的loss= 5.818386554718018
目前的loss= 5.786195755004883
目前的loss= 5.777205467224121
目前的loss= 5.7423415184021
目前的loss= 5.709352493286133
目前的loss= 5.695651531219482


In [21]:
# 统计版：数出每个 (前字, 后字) 组合的计数,这个可以算出来理想标准的loss应该是多少
import torch
N = torch.zeros((3378, 3378), dtype=torch.int32)
for x, y in zip(xs.tolist(), ys.tolist()):
    N[x, y] += 1

P = (N + 1).float()          # +1 平滑，老朋友了
P /= P.sum(1, keepdim=True)  # 每行归一化成概率

# 统计版的 NLL：每个样本，查它正确答案的概率，取 -log，求平均
loss_stat = -P[xs, ys].log().mean()
print(loss_stat)

tensor(5.4152)


In [23]:
#接下来就是检验的时候了
out=[]
ix=0
for _ in range(10):
    #一样的,不过这里直接查表不然就炸了
    logits=W[ix]
    counts=logits.exp()
    p=counts/counts.sum()
    ix=torch.multinomial(p,num_samples=1,replacement=True,generator=g).item()
    out.append(itos[ix])
    if ix==0:
        break
print(''.join(out))


孫輸璂獨壹摶璇.


可以看结果就知道，中文的这个还是比起英文差不少。因为英文就26个字母，中文的汉字太多，采用一样的思路的话效果远不及英文
而且相对来说中文版本的本身loss应该在5.4，而英文只有2.6，这已经是天差地别了。而且中文版本大概只能收敛到这个地步了，再进一步也不太可能了。这差不多就是中文bigram所能做到的极限了